## AUTODOC - Hometask

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

df = pd.read_csv("data_set_da_test.csv")
df.head()

,event_date,session,user,page_type,event_type,product
0,2022-10-08 17:02:41,14274187577460658115s,2006979063809820329u,search_listing_page,page_view,0
1,2022-10-08 17:06:19,14274187577460658115s,2006979063809820329u,search_listing_page,page_view,0
2,2022-10-08 22:19:47,2704204808571844605s,2007646148110679693u,listing_page,page_view,0
3,2022-10-08 22:24:30,8970170322512311099s,11839491588321754710u,search_listing_page,page_view,0
4,2022-10-08 21:22:20,16223970371660715740s,11839887495958431209u,product_page,page_view,0


### First Data Validations

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 637238 entries, 0 to 637237
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   event_date  637238 non-null  object
 1   session     637238 non-null  object
 2   user        637238 non-null  object
 3   page_type   637238 non-null  object
 4   event_type  637238 non-null  object
 5   product     637238 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 29.2+ MB


In [77]:
df.shape

(637238, 6)

In [78]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
event_date,637238,464539,2022-10-05 11:57:23,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session,637238,340443,9543669642136985868s,1264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
user,637238,288088,11769744300065907078u,1266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
page_type,637238,4,product_page,282950,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event_type,637238,3,page_view,612498,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product,"637,238",NaN,NaN,NaN,"714,635","4,536,790",0,0,0,0,"38,436,515"


In [79]:
null_summary = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_pct": df.isna().mean() * 100,
})

null_summary

,null_count,null_pct
event_date,0,0
session,0,0
user,0,0
page_type,0,0
event_type,0,0
product,0,0


In [80]:
## Understanding categorical columns

categorical_cols = ["page_type", "event_type"]

for col in categorical_cols:
    print(f"\n--- {col.upper()} ---")
    display(df[col].value_counts(dropna=False).head(20))


--- PAGE_TYPE ---


page_type
product_page           282950
listing_page           231789
search_listing_page    113758
order_page               8741
Name: count, dtype: int64


--- EVENT_TYPE ---


event_type
page_view      612498
add_to_cart     15999
order            8741
Name: count, dtype: int64

In [81]:
pd.set_option('display.float_format', '{:,.0f}'.format)
print(df["product"].dtype)
print(df["product"].describe())

print(f"Número de produtos únicos: {df['product'].nunique()}")
print(f"Total de linhas: {len(df)}")

int64
count      637,238
mean       714,635
std      4,536,790
min              0
25%              0
50%              0
75%              0
max     38,436,515
Name: product, dtype: float64
Número de produtos únicos: 12442
Total de linhas: 637238


In [14]:
pd.crosstab(
    index=df["event_type"],
    columns=df["page_type"]
)

page_type,listing_page,order_page,product_page,search_listing_page
event_type,,,,
add_to_cart,526,0,12200,3273
order,0,8741,0,0
page_view,231263,0,270750,110485


### Product Column Explore

In [12]:
cart_df = df[
    (df["event_type"] == "add_to_cart") &
    (df["product"].notna()) &
    (df["product"] != 0)
].copy()

In [16]:
product_cart_counts = (
    cart_df.groupby("product")
    .agg(
        add_to_cart_events=("event_type", "count"),
        users=("user", "nunique"),
        sessions=("session", "nunique")
    )
    .reset_index()
    .sort_values("add_to_cart_events", ascending=False)
)

products_added_more_than_once = product_cart_counts.query("add_to_cart_events > 1")

print("Total products added:", cart_df["product"].nunique())
print("Products added more than once:", products_added_more_than_once["product"].nunique())
print("Share product add more than once:", products_added_more_than_once["product"].nunique() / cart_df["product"].nunique())

products_added_more_than_once.head(10)

Total products added: 12441
Products added more than once: 2193
Share product add more than once: 0.17627200385821076


,product,add_to_cart_events,users,sessions
3460,26372760,21,21,21
5288,27133309,20,20,20
4920,27075197,18,1,1
4184,26780006,15,6,6
1350,20283956,13,11,11
735,20053168,13,13,13
5666,27189395,12,9,9
5360,27147563,12,2,2
7762,31845429,12,12,12
11364,35710698,11,2,2


In [19]:
product_cart_by_session = (
    cart_df.groupby(["session", "product"])
    .agg(
        add_to_cart_events=("event_type", "count"),
        user=("user", "first")
    )
    .reset_index()
)

same_product_added_twice_session = (
    product_cart_by_session
    .query("add_to_cart_events > 1")
    .sort_values("add_to_cart_events", ascending=False)
)

# Total de sessões que tiveram pelo menos um add_to_cart
total_sessions_with_cart = cart_df["session"].nunique()

# Sessões que adicionaram o mesmo produto mais de uma vez
sessions_repeated_product = same_product_added_twice_session["session"].nunique()

print("Total sessions with add_to_cart:", total_sessions_with_cart)
print("Sessions with repeated add_to_cart of the same product:", sessions_repeated_product)
print("Share:", sessions_repeated_product / total_sessions_with_cart)

same_product_added_twice_session.head(20)

Total sessions with add_to_cart: 10667
Sessions with repeated add_to_cart of the same product: 1002
Share: 0.09393456454485798


,session,product,add_to_cart_events,user
4597,15278546023749867344s,27075197,18,77095704668614000u
6860,17821203209891093616s,27147563,11,13473897499838435703u
3515,14078220285313645328s,27857522,11,6572634401610340280u
374,10407914825077295011s,35710698,10,14994340849643809483u
13998,9299248968106933623s,21133893,10,3790902706967614872u
12577,7643021673926988654s,27785528,9,17785870066159206027u
6925,17891144200139890760s,20330628,8,7716529702735234952u
8052,2605832711525718719s,27127082,7,4399416701106629910u
10908,5680064279292765904s,36953467,7,6905391506811269383u
11153,5961177114271039512s,30339917,7,16873796923125700708u


In [18]:
case_session = same_product_added_twice_session.iloc[0]["session"]
case_product = same_product_added_twice_session.iloc[0]["product"]

df[
    (df["session"] == case_session) &
    (df["product"] == case_product)
].sort_values("event_date")

,event_date,session,user,page_type,event_type,product,event_day
193091,2022-10-09 13:07:13,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
608992,2022-10-09 13:07:18,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
608991,2022-10-09 13:07:21,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
608993,2022-10-09 13:07:22,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
171492,2022-10-09 13:07:22,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
608987,2022-10-09 13:07:22,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
317629,2022-10-09 13:07:22,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
608989,2022-10-09 13:07:23,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
171491,2022-10-09 13:07:23,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09
193092,2022-10-09 13:07:23,15278546023749867344s,77095704668614000u,search_listing_page,add_to_cart,27075197,2022-10-09


In [20]:
distribution = (
    product_cart_by_session["add_to_cart_events"]
    .value_counts()
    .sort_index()
    .reset_index()
)

distribution.columns = ["add_to_cart_events", "session_product_count"]

distribution["share"] = (
    distribution["session_product_count"] /
    distribution["session_product_count"].sum()
)

distribution["share"] = (
    distribution["share"] * 100
).round(2)

distribution

,add_to_cart_events,session_product_count,share
0,1,13568,92.91
1,2,855,5.85
2,3,104,0.71
3,4,38,0.26
4,5,13,0.09
5,6,11,0.08
6,7,7,0.05
7,8,1,0.01
8,9,1,0.01
9,10,2,0.01


In [82]:
products = (
    df["product"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

products.diff().describe()

count       12,441
mean         3,090
std        172,605
min              1
25%             18
50%            105
75%            498
max     19,103,284
Name: product, dtype: float64

In [83]:
zero_summary = (df == 0).sum().sort_values(ascending=False)

zero_summary_pct = ((df == 0).mean() * 100).sort_values(ascending=False)

pd.DataFrame({
    "zero_count": zero_summary,
    "zero_pct": zero_summary_pct
})

,zero_count,zero_pct
product,621239,97
event_date,0,0
session,0,0
user,0,0
page_type,0,0
event_type,0,0


In [84]:
pd.crosstab(
    df["page_type"],
    df["product"] == 0,
    margins=True
)

product,False,True,All
page_type,,,
listing_page,526,231263,231789
order_page,0,8741,8741
product_page,12200,270750,282950
search_listing_page,3273,110485,113758
All,15999,621239,637238


In [85]:
pd.crosstab(
    df["event_type"],
    df["product"] == 0,
    margins=True
)

product,False,True,All
event_type,,,
add_to_cart,15999,0,15999
order,0,8741,8741
page_view,0,612498,612498
All,15999,621239,637238


In [86]:
df[df["product"] != 0].head(20)

,event_date,session,user,page_type,event_type,product
27,2022-10-12 08:48:30,16454924377300391746s,14844239410760899440u,product_page,add_to_cart,33842734
28,2022-10-12 08:47:29,16454924377300391746s,14844239410760899440u,product_page,add_to_cart,33849131
54,2022-10-02 16:13:29,14791989132776030059s,3475228846498346303u,product_page,add_to_cart,26683722
152,2022-10-12 16:39:27,9240904522911146426s,7151757536501293113u,product_page,add_to_cart,32727128
173,2022-10-06 16:31:16,16351345959988978592s,7011174022396541275u,product_page,add_to_cart,20968309
236,2022-10-10 18:11:04,15413017811083664755s,8737029530618806639u,product_page,add_to_cart,36391982
248,2022-10-01 18:33:36,1149922820988919328s,6535321412024345u,search_listing_page,add_to_cart,20231011
249,2022-10-01 18:52:42,1149922820988919328s,6535321412024345u,search_listing_page,add_to_cart,26622758
250,2022-10-01 18:27:51,1149922820988919328s,6535321412024345u,search_listing_page,add_to_cart,19630378
261,2022-10-08 13:46:52,536717885223711276s,2034694364905325696u,product_page,add_to_cart,30894373


### Session Validations

In [88]:
user_sessions = (
    df.groupby("user")["session"]
      .nunique()
      .reset_index(name="num_sessions")
)

user_sessions[user_sessions["num_sessions"] > 1]

,user,num_sessions
17,10001170501335904343u,2
27,10001624294095919955u,3
29,10001657122833638878u,2
32,10001743582852900931u,2
49,10002703279131110244u,2
...,...,...
288040,9997079536040347097u,2
288046,999743330616179986u,8
288052,9997899854331153682u,4
288067,9998569173007604481u,13


In [89]:
(user_sessions["num_sessions"] > 1).sum()

np.int64(29790)

In [90]:
user_sessions = (
    df[["user", "session"]]
    .drop_duplicates()
    .groupby("user")["session"]
    .count()
    .reset_index(name="num_sessions")
)

session_distribution = (
    user_sessions["num_sessions"]
    .value_counts()
    .sort_index()
    .reset_index()
)

session_distribution.columns = ["num_sessions", "num_users"]

session_distribution["pct_users"] = (
    session_distribution["num_users"]
    / session_distribution["num_users"].sum()
    * 100
).round(2)

session_distribution

,num_sessions,num_users,pct_users
0,1,258298,90
1,2,20540,7
2,3,5051,2
3,4,1820,1
4,5,903,0
5,6,476,0
6,7,275,0
7,8,170,0
8,9,115,0
9,10,104,0


### User Validations

In [15]:
# Usuários com mais de 1 evento
user_events = (
    df.groupby("user")
    .size()
    .reset_index(name="total_events")
)

valid_users = user_events[
    user_events["total_events"] > 1
]["user"]

# Sessões por usuário, removendo usuários com apenas 1 evento
sessions_per_user = (
    df[df["user"].isin(valid_users)]
    .groupby("user")["session"]
    .nunique()
    .reset_index(name="number_sessions")
)

# Distribuição
sessions_distribution = (
    sessions_per_user
    .groupby("number_sessions")
    .size()
    .reset_index(name="users")
)

sessions_distribution["share_users"] = (
    sessions_distribution["users"] /
    sessions_distribution["users"].sum() * 100
).round(2)

sessions_distribution.head(20)

,number_sessions,users,share_users
0,1,61282,67.29
1,2,20540,22.55
2,3,5051,5.55
3,4,1820,2.00
4,5,903,0.99
5,6,476,0.52
6,7,275,0.30
7,8,170,0.19
8,9,115,0.13
9,10,104,0.11


In [91]:
session_users = (
    df.groupby("session")["user"]
      .nunique()
      .reset_index(name="num_users")
)

sessions_with_multiple_users = session_users[
    session_users["num_users"] > 1
]

sessions_with_multiple_users

,session,num_users


### User that generated more than one session per day

In [11]:
# Garantir datetime
df["event_date"] = pd.to_datetime(df["event_date"])

# Extrair o dia
df["event_day"] = df["event_date"].dt.date

# Número de sessões por usuário por dia
sessions_per_day = (
    df.groupby(["user", "event_day"])["session"]
      .nunique()
      .reset_index(name="sessions_per_day")
)

# Usuários que tiveram mais de uma sessão em algum dia
users_multiple_sessions = (
    sessions_per_day
    .query("sessions_per_day > 1")["user"]
    .unique()
)

# Estatísticas
total_users = df["user"].nunique()
users_with_multiple_sessions = len(users_multiple_sessions)

print(f"Total users: {total_users:,}")
print(f"Users with >1 session in at least one day: {users_with_multiple_sessions:,}")
print(f"Percentage: {users_with_multiple_sessions / total_users:.2%}")

sessions_per_day.query("sessions_per_day > 1").head(10)

Total users: 288,088
Users with >1 session in at least one day: 14,976
Percentage: 5.20%


,user,event_day,sessions_per_day
17,10001170501335904343u,2022-10-11,2
27,10001624294095919955u,2022-10-05,2
30,10001657122833638878u,2022-10-10,2
33,10001743582852900931u,2022-10-01,2
50,10002703279131110244u,2022-10-03,2
128,10006497281265404039u,2022-10-09,2
168,10008383400758209909u,2022-10-11,2
185,10008841403698122831u,2022-10-05,3
212,10009897836937049107u,2022-10-01,2
234,100105731458701234u,2022-10-09,3


### Users more than one day exploring

In [4]:
df["event_date"] = pd.to_datetime(df["event_date"])
df["event_day"] = df["event_date"].dt.date

user_days = (
    df.groupby("user")
      .agg(
          distinct_days=("event_day", "nunique"),
          distinct_sessions=("session", "nunique"),
          first_event=("event_date", "min"),
          last_event=("event_date", "max")
      )
      .reset_index()
)

users_multiple_days = user_days[user_days["distinct_days"] > 1]

users_multiple_days.sort_values("distinct_days", ascending=False).head(20)

,user,distinct_days,distinct_sessions,first_event,last_event
260863,8428431602095059916u,14,45,2022-09-30 09:14:38,2022-10-13 18:08:25
258632,8299217561796218936u,14,19,2022-09-30 09:14:08,2022-10-13 06:23:08
145426,18375178374461269577u,14,14,2022-09-30 06:37:50,2022-10-13 06:45:54
139332,18020998732636818491u,14,75,2022-09-30 06:27:19,2022-10-13 16:40:28
161819,2718409947134398885u,14,31,2022-09-30 11:05:09,2022-10-13 10:42:59
179339,3731573923359431787u,14,48,2022-09-30 06:13:34,2022-10-13 17:13:17
277832,9398822402354953u,14,27,2022-09-30 15:27:08,2022-10-13 14:44:27
64740,13730386746524059311u,14,21,2022-09-30 12:07:58,2022-10-13 09:22:22
50621,12911211005900884833u,14,46,2022-09-30 08:21:38,2022-10-13 07:13:14
109887,16320078228916101933u,14,15,2022-09-30 20:48:54,2022-10-13 19:05:26


### Sessions that cross days

In [5]:
# Garantir datetime
df["event_date"] = pd.to_datetime(df["event_date"])

# Extrair apenas a data
df["event_day"] = df["event_date"].dt.date

# Quantos dias distintos cada sessão aparece
session_days = (
    df.groupby("session")
      .agg(
          distinct_days=("event_day", "nunique"),
          first_day=("event_day", "min"),
          last_day=("event_day", "max"),
          users=("user", "nunique")
      )
      .reset_index()
)

# Sessões presentes em mais de um dia
sessions_multiple_days = session_days.query("distinct_days > 1")

print(f"Total sessions: {len(session_days):,}")
print(f"Sessions in multiple days: {len(sessions_multiple_days):,}")
print(f"Percentage: {len(sessions_multiple_days)/len(session_days):.2%}")

sessions_multiple_days.head()

Total sessions: 340,443
Sessions in multiple days: 206
Percentage: 0.06%


,session,distinct_days,first_day,last_day,users
782,10038902305023999556s,2,2022-09-30,2022-10-01,1
5118,10245152817514270896s,2,2022-10-05,2022-10-06,1
5261,10252185668266026842s,2,2022-10-02,2022-10-03,1
6947,10334951306497501346s,2,2022-10-03,2022-10-04,1
8220,10395959021605590354s,2,2022-10-01,2022-10-02,1


### First session events

In [92]:
df = df.sort_values(["session", "event_date"])

df["session_rank"] = (
    df.groupby("session")
      .cumcount() + 1
)

df = df.sort_values(["user", "event_date"])

df["user_rank"] = (
    df.groupby("user")
      .cumcount() + 1
)

df.head(20)

,event_date,session,user,page_type,event_type,product,session_rank,user_rank
452151,2022-10-02 22:33:00,18233814593797186025s,10000015204044662882u,product_page,page_view,0,1,1
633324,2022-09-30 10:25:03,13458793695112618041s,10000054772579221757u,product_page,page_view,0,1,1
475998,2022-10-10 19:07:13,4338120812505723853s,10000059160930102536u,listing_page,page_view,0,1,1
570514,2022-10-10 19:07:43,4338120812505723853s,10000059160930102536u,search_listing_page,page_view,0,2,2
475999,2022-10-10 19:09:21,4338120812505723853s,10000059160930102536u,product_page,page_view,0,3,3
475531,2022-10-01 20:09:10,2474253138883643460s,10000070603737986791u,product_page,page_view,0,1,1
413620,2022-10-11 06:44:54,6210557599463868535s,10000178367361043064u,product_page,page_view,0,1,1
270686,2022-09-30 11:10:39,2888515939535674785s,10000217975699300852u,product_page,page_view,0,1,1
396624,2022-10-02 14:54:00,14495504031420244331s,10000284942168687885u,listing_page,page_view,0,1,1
420153,2022-10-02 15:44:19,1264365435103854899s,1000029122569142942u,listing_page,page_view,0,1,1


In [94]:
first_session_event = df[df["session_rank"] == 1]

pd.crosstab(
    index=first_session_event["page_type"],
    columns=first_session_event["event_type"],
    margins=True
)

event_type,add_to_cart,order,page_view,All
page_type,,,,
listing_page,6,0,176826,176832
order_page,0,1399,0,1399
product_page,153,0,146771,146924
search_listing_page,11,0,15277,15288
All,170,1399,338874,340443


### First page validation

In [42]:
df["event_date"] = pd.to_datetime(df["event_date"])

first_page_by_session = (
    df.sort_values(["session", "event_date"])
      .groupby("session", as_index=False)
      .first()[["session", "event_type", "page_type"]]
      .rename(columns={
          "event_type": "first_event",
          "page_type": "first_page"
      })
)

first_page_by_session["first_page"] = (
    first_page_by_session["first_page"]
    .fillna("no_page_type")
)

first_event_distribution = (
    first_page_by_session
    .groupby(["first_event", "first_page"], dropna=False)
    .size()
    .reset_index(name="sessions")
    .sort_values("sessions", ascending=False)
)

first_event_distribution["share"] = (
    first_event_distribution["sessions"] /
    first_event_distribution["sessions"].sum()
).round(4)

first_event_distribution

,first_event,first_page,sessions,share
4,page_view,listing_page,176826,0.5194
5,page_view,product_page,146771,0.4311
6,page_view,search_listing_page,15277,0.0449
3,order,order_page,1399,0.0041
1,add_to_cart,product_page,153,0.0004
2,add_to_cart,search_listing_page,11,0.0000
0,add_to_cart,listing_page,6,0.0000


### Exploring event_date

In [95]:
df["event_date"] = pd.to_datetime(df["event_date"])

print(f"Min event_date: {df['event_date'].min()}")
print(f"Max event_date: {df['event_date'].max()}")

print(f"Period: {df['event_date'].max() - df['event_date'].min()}")

Min event_date: 2022-09-30 00:00:00
Max event_date: 2022-10-13 23:59:54
Period: 13 days 23:59:54


In [96]:
### Section Duration

session_duration = (
    df.groupby("session")
      .agg(
          start_time=("event_date", "min"),
          end_time=("event_date", "max")
      )
      .reset_index()
)

session_duration["duration"] = (
    session_duration["end_time"] - session_duration["start_time"]
)

session_duration["duration"].describe()

count                       340443
mean     0 days 00:01:55.524742761
std      0 days 00:07:21.989608111
min                0 days 00:00:00
25%                0 days 00:00:00
50%                0 days 00:00:00
75%                0 days 00:00:05
max                0 days 05:51:53
Name: duration, dtype: object

In [97]:
### User Duration

user_duration = (
    df.groupby("user")
      .agg(
          start_time=("event_date", "min"),
          end_time=("event_date", "max")
      )
      .reset_index()
)

user_duration["duration"] = (
    user_duration["end_time"] - user_duration["start_time"]
)

user_duration["duration"].describe()

count                       288088
mean     0 days 06:40:52.129408375
std      1 days 07:28:33.931087019
min                0 days 00:00:00
25%                0 days 00:00:00
50%                0 days 00:00:00
75%                0 days 00:00:49
max               13 days 23:22:06
Name: duration, dtype: object

### Behaviour Model 

In [126]:
# Flag de order
df["is_order"] = df["event_type"].eq("order")

# Orders por sessão
orders_per_session = (
    df.groupby("session")["is_order"]
      .sum()
      .reset_index(name="num_orders")
)

orders_session_distribution = (
    orders_per_session["num_orders"]
      .value_counts()
      .sort_index()
      .reset_index()
)

orders_session_distribution.columns = ["num_orders", "num_sessions"]

orders_session_distribution["pct_sessions"] = (
    orders_session_distribution["num_sessions"]
    / orders_session_distribution["num_sessions"].sum()
    * 100
).round(2)

orders_session_distribution

,num_orders,num_sessions,pct_sessions
0,0,332806,98
1,1,6944,2
2,2,442,0
3,3,161,0
4,4,56,0
5,5,15,0
6,6,11,0
7,7,5,0
8,8,1,0
9,10,1,0


In [128]:
orders_session_distribution_nonzero = (
    orders_per_session
    .query("num_orders > 0")["num_orders"]
    .value_counts()
    .sort_index()
    .reset_index()
)

orders_session_distribution_nonzero.columns = ["num_orders", "num_sessions"]

orders_session_distribution_nonzero["pct_sessions"] = (
    orders_session_distribution_nonzero["num_sessions"]
    / orders_session_distribution_nonzero["num_sessions"].sum()
    * 100
).round(2)

orders_session_distribution_nonzero

,num_orders,num_sessions,pct_sessions
0,1,6944,91
1,2,442,6
2,3,161,2
3,4,56,1
4,5,15,0
5,6,11,0
6,7,5,0
7,8,1,0
8,10,1,0
9,12,1,0


In [3]:
df["is_order"] = df["event_type"].eq("order")

# Orders por usuário
orders_per_user = (
    df.groupby("user")["is_order"]
      .sum()
      .reset_index(name="num_orders")
)

orders_user_distribution = (
    orders_per_user["num_orders"]
      .value_counts()
      .sort_index()
      .reset_index()
)

orders_user_distribution.columns = ["num_orders", "num_users"]

orders_user_distribution["pct_users"] = (
    orders_user_distribution["num_users"]
    / orders_user_distribution["num_users"].sum()
    * 100
).round(2)

orders_user_distribution

,num_orders,num_users,pct_users
0,0,280750,97.45
1,1,6480,2.25
2,2,560,0.19
3,3,178,0.06
4,4,70,0.02
5,5,21,0.01
6,6,14,0.00
7,7,4,0.00
8,8,5,0.00
9,9,2,0.00


In [129]:
orders_user_distribution_nonzero = (
    orders_per_user
    .query("num_orders > 0")["num_orders"]
    .value_counts()
    .sort_index()
    .reset_index()
)

orders_user_distribution_nonzero.columns = ["num_orders", "num_users"]

orders_user_distribution_nonzero["pct_users"] = (
    orders_user_distribution_nonzero["num_users"]
    / orders_user_distribution_nonzero["num_users"].sum()
    * 100
).round(2)

orders_user_distribution_nonzero

,num_orders,num_users,pct_users
0,1,6480,88
1,2,560,8
2,3,178,2
3,4,70,1
4,5,21,0
5,6,14,0
6,7,4,0
7,8,5,0
8,9,2,0
9,12,3,0


In [20]:
# Datetime + ordenação
df["event_date"] = pd.to_datetime(df["event_date"])
df = df.sort_values(["session", "event_date"]).reset_index(drop=True)

# Flags auxiliares
df["is_page_view"] = (df["event_type"] == "page_view").astype(int)
df["is_add_to_cart"] = (df["event_type"] == "add_to_cart").astype(int)
df["is_order"] = (df["event_type"] == "order").astype(int)
df["is_product_page_view"] = (
    (df["page_type"] == "product_page") &
    (df["event_type"] == "page_view")
).astype(int)

# Para contar page_types apenas quando houve page_view
df["page_type_if_pageview"] = df["page_type"].where(df["event_type"] == "page_view")

# Ordem dos eventos dentro da sessão
df["event_order"] = df.groupby("session").cumcount() + 1

# Tabela base por sessão
session_model = (
    df.groupby("session", sort=False)
      .agg(
          user=("user", "first"),
          session_start=("event_date", "min"),
          session_end=("event_date", "max"),

          generated_order=("is_order", "max"),

          number_pageviews=("is_page_view", "sum"),
          has_add_to_cart=("is_add_to_cart", "max"),
          number_add_to_cart=("is_add_to_cart", "sum"),

          different_page_types_viewed=("page_type_if_pageview", "nunique"),
          product_page_views=("is_product_page_view", "sum")
      )
      .reset_index()
)

first_page = (
    df[df["page_type"] != "order_page"]
      .sort_values(["session", "event_date"])
      .groupby("session", sort=False)
      .first()[["page_type"]]
      .rename(columns={"page_type": "first_page"})
      .reset_index()
)

# Primeiro add_to_cart por sessão
first_cart = (
    df[df["is_add_to_cart"] == 1]
      .groupby("session", sort=False)["event_order"]
      .min()
      .reset_index(name="first_add_to_cart_position")
)

session_model = session_model.merge(first_cart, on="session", how="left")
session_model = (
    session_model
    .merge(first_page, on="session", how="left")
)

# Pageviews até o primeiro add_to_cart
df_with_cart = df.merge(first_cart, on="session", how="left")

pageviews_before_cart = (
    df_with_cart[
        (df_with_cart["is_page_view"] == 1) &
        (df_with_cart["event_order"] < df_with_cart["first_add_to_cart_position"])
    ]
    .groupby("session", sort=False)
    .size()
    .reset_index(name="pageviews_before_first_add_to_cart")
)

session_model = session_model.merge(pageviews_before_cart, on="session", how="left")

# Para sessões sem add_to_cart, preencher com total de pageviews
session_model["pageviews_before_first_add_to_cart"] = (
    session_model["pageviews_before_first_add_to_cart"]
    .fillna(session_model["number_pageviews"])
)

session_model["first_add_to_cart_position"] = session_model["first_add_to_cart_position"].fillna(0)

# Add_to_cart repetido do mesmo produto dentro da sessão
cart_df = df[
    (df["is_add_to_cart"] == 1) &
    (df["product"].notna()) &
    (df["product"] != 0)
].copy()

product_cart_by_session = (
    cart_df.groupby(["session", "product"], sort=False)
      .size()
      .reset_index(name="same_product_cart_count")
)

repeated_product_features = (
    product_cart_by_session
      .assign(is_repeated_product=lambda x: (x["same_product_cart_count"] > 1).astype(int))
      .groupby("session", sort=False)
      .agg(
          max_same_product_add_to_cart=("same_product_cart_count", "max"),
          repeated_products_count=("is_repeated_product", "sum"),
          has_repeated_product_add_to_cart=("is_repeated_product", "max")
      )
      .reset_index()
)

session_model = session_model.merge(repeated_product_features, on="session", how="left")

session_model[[
    "max_same_product_add_to_cart",
    "repeated_products_count",
    "has_repeated_product_add_to_cart"
]] = session_model[[
    "max_same_product_add_to_cart",
    "repeated_products_count",
    "has_repeated_product_add_to_cart"
]].fillna(0)

session_model.head()

,session,user,session_start,session_end,generated_order,number_pageviews,has_add_to_cart,number_add_to_cart,different_page_types_viewed,product_page_views,first_add_to_cart_position,first_page,pageviews_before_first_add_to_cart,max_same_product_add_to_cart,repeated_products_count,has_repeated_product_add_to_cart
0,10000000052138714153s,8035194775607769271u,2022-10-12 16:26:46,2022-10-12 16:26:46,0,1,0,0,1,0,0.0,search_listing_page,1.0,0.0,0.0,0.0
1,10000003251787879077s,2613381957342920411u,2022-10-01 19:16:18,2022-10-01 19:16:56,0,2,0,0,1,0,0.0,search_listing_page,2.0,0.0,0.0,0.0
2,10000028460721229362s,12451948957440700749u,2022-10-13 09:08:27,2022-10-13 09:08:33,0,2,0,0,1,0,0.0,listing_page,2.0,0.0,0.0,0.0
3,10000145746398845278s,16615957149200537975u,2022-10-04 12:59:05,2022-10-04 12:59:05,0,1,0,0,1,0,0.0,listing_page,1.0,0.0,0.0,0.0
4,1000015740551295390s,6456170714504168075u,2022-10-03 16:03:53,2022-10-03 16:03:53,0,1,0,0,1,0,0.0,listing_page,1.0,0.0,0.0,0.0


In [21]:
session_model.describe().T

,count,mean,min,25%,50%,75%,max,std
session_start,340443,2022-10-06 20:44:39.534720768,2022-09-30 00:00:00,2022-10-03 07:56:22.500000,2022-10-06 12:52:27,2022-10-10 16:48:14,2022-10-13 23:59:54,NaN
session_end,340443,2022-10-06 20:46:35.059463680,2022-09-30 00:00:00,2022-10-03 07:57:23.500000,2022-10-06 12:54:24,2022-10-10 16:49:48,2022-10-13 23:59:54,NaN
generated_order,340443.0,0.022433,0.0,0.0,0.0,0.0,1.0,0.148086
number_pageviews,340443.0,1.799121,0.0,1.0,1.0,2.0,1264.0,3.557119
has_add_to_cart,340443.0,0.031333,0.0,0.0,0.0,0.0,1.0,0.174216
number_add_to_cart,340443.0,0.046995,0.0,0.0,0.0,0.0,18.0,0.323369
different_page_types_viewed,340443.0,1.105004,0.0,1.0,1.0,1.0,3.0,0.362395
product_page_views,340443.0,0.795287,0.0,0.0,0.0,1.0,112.0,1.501062
first_add_to_cart_position,340443.0,0.128468,0.0,0.0,0.0,0.0,60.0,0.939257
pageviews_before_first_add_to_cart,340443.0,1.700711,0.0,1.0,1.0,2.0,1264.0,3.302298


Understand funnel events that will be good to a session conversion
- Gerou order (target value)
- Number of pageview events
- If there is any add_to_cart generated
- Number of add_to_cart events
- Which page is the first in the session
- How many different type of pages generated page view
- How many product pages was viewed during the session
- Event position of the first add_to_card 
- How many page views until the first add_to_card
- How many add_to_cart same product max times
- How many products has more than one add_to_card event
- Flag if has or not repeated products add_to_cart

In [22]:
features = [
    "number_pageviews",
    "has_add_to_cart",
    "number_add_to_cart",
    "first_page",
    "different_page_types_viewed",
    "product_page_views",
    "pageviews_before_first_add_to_cart",
    "repeated_products_count",
    "has_repeated_product_add_to_cart"
]

target = "generated_order"

model_df = session_model[features + [target]].copy()

In [23]:
model_df = pd.get_dummies(
    model_df,
    columns=["first_page"]
)

model_df.head()

,number_pageviews,has_add_to_cart,number_add_to_cart,different_page_types_viewed,product_page_views,pageviews_before_first_add_to_cart,repeated_products_count,has_repeated_product_add_to_cart,generated_order,first_page_listing_page,first_page_product_page,first_page_search_listing_page
0,1,0,0,1,0,1.0,0.0,0.0,0,False,False,True
1,2,0,0,1,0,2.0,0.0,0.0,0,False,False,True
2,2,0,0,1,0,2.0,0.0,0.0,0,True,False,False
3,1,0,0,1,0,1.0,0.0,0.0,0,True,False,False
4,1,0,0,1,0,1.0,0.0,0.0,0,True,False,False


In [24]:
X = model_df.drop(columns=target)
y = model_df[target]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score

pred = rf.predict_proba(X_test)[:,1]

print(
    "AUC:",
    roc_auc_score(y_test, pred)
)

importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

importance

AUC: 0.8905520941332612


,feature,importance
2,number_add_to_cart,0.248439
0,number_pageviews,0.179355
5,pageviews_before_first_add_to_cart,0.167572
1,has_add_to_cart,0.164870
3,different_page_types_viewed,0.129919
4,product_page_views,0.079322
6,repeated_products_count,0.008048
9,first_page_product_page,0.006206
8,first_page_listing_page,0.005683
10,first_page_search_listing_page,0.005306


In [25]:
## Number add_to_card

(
    session_model
    .groupby("number_add_to_cart")
    .agg(
        sessions=("session", "count"),
        conversion_rate=("generated_order", "mean")
    )
    .assign(
        conversion_rate=lambda x: (100*x["conversion_rate"]).round(2),
        share=lambda x: (100*x["sessions"]/x["sessions"].sum()).round(2)
    )
)

,sessions,conversion_rate,share
number_add_to_cart,,,
0,329776,1.03,96.87
1,7592,37.38,2.23
2,1962,45.57,0.58
3,583,46.83,0.17
4,263,47.15,0.08
5,114,46.49,0.03
6,72,48.61,0.02
7,38,39.47,0.01
8,16,37.50,0.00


In [46]:
## Has add to cart

(
    session_model
    .groupby("has_add_to_cart")
    .agg(
        sessions=("session","count"),
        conversion_rate=("generated_order","mean")
    )
    .assign(
        conversion_rate=lambda x:(100*x["conversion_rate"]).round(2),
        share=lambda x:(100*x["sessions"]/x["sessions"].sum()).round(2)
    )
)

,sessions,conversion_rate,share
has_add_to_cart,,,
0,329776,1.03,96.87
1,10667,39.83,3.13


In [50]:
## Number of pageviews

session_model["pageview_bucket"] = pd.cut(
    session_model["number_pageviews"],
    bins=[0,2,5,10,20,999],
    labels=[
        "1-2",
        "3-5",
        "6-10",
        "11-20",
        "20+"
    ]
)

(
    session_model
    .groupby("pageview_bucket")
    .agg(
        sessions=("session","count"),
        conversion_rate=("generated_order","mean")
    )
    .assign(
        conversion_rate=lambda x:(100*x["conversion_rate"]).round(2),
        share=lambda x:(100*x["sessions"]/x["sessions"].sum()).round(2)
    )
)

/tmp/ipykernel_11784/3413079427.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("pageview_bucket")


,sessions,conversion_rate,share
pageview_bucket,,,
1-2,292831,0.88,86.29
3-5,31662,6.14,9.33
6-10,10037,12.08,2.96
11-20,3677,16.48,1.08
20+,1137,20.84,0.34


In [ ]:
### Different first page

(
    session_model
    .groupby("first_page")
    .agg(
        sessions=("session","count"),
        conversion_rate=("generated_order","mean")
    )
    .assign(
        conversion_rate=lambda x:(100*x["conversion_rate"]).round(2),
        share=lambda x:(100*x["sessions"]/x["sessions"].sum()).round(2)
    )
)

,sessions,conversion_rate,share
first_page,,,
listing_page,176832,1.33,51.94
order_page,1399,100.00,0.41
product_page,146924,2.27,43.16
search_listing_page,15288,3.53,4.49


In [26]:
### pageviews before first add to cart

session_model["pv_before_cart_bucket"] = pd.cut(
    session_model["pageviews_before_first_add_to_cart"],
    bins=[-1,0,1,2,5,10,999],
    labels=[
        "0",
        "1",
        "2",
        "3-5",
        "6-10",
        "10+"
    ]
)

(
    session_model
    .groupby("pv_before_cart_bucket")
    .agg(
        sessions=("session","count"),
        conversion_rate=("generated_order","mean")
    )
    .assign(
        conversion_rate=lambda x:(100*x["conversion_rate"]).round(2),
        share=lambda x:(100*x["sessions"]/x["sessions"].sum()).round(2)
    )
)

/tmp/ipykernel_36716/1971765422.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("pv_before_cart_bucket")


,sessions,conversion_rate,share
pv_before_cart_bucket,,,
0,1098,96.45,0.32
1,253837,0.87,74.56
2,41957,3.64,12.32
3-5,31236,5.94,9.18
6-10,8793,7.88,2.58
10+,3521,8.01,1.03


In [58]:
different_pages_conversion = (
    session_model
    .groupby("different_page_types_viewed")
    .agg(
        sessions=("session", "count"),
        orders=("generated_order", "sum"),
        conversion_rate=("generated_order", "mean")
    )
    .reset_index()
)

different_pages_conversion["share_sessions"] = (
    different_pages_conversion["sessions"] /
    different_pages_conversion["sessions"].sum() * 100
).round(2)

different_pages_conversion["conversion_rate"] = (
    different_pages_conversion["conversion_rate"] * 100
).round(2)

different_pages_conversion

,different_page_types_viewed,sessions,orders,conversion_rate,share_sessions
0,0,1098,1059,96.45,0.32
1,1,307759,3107,1.01,90.40
2,2,26326,2443,9.28,7.73
3,3,5260,1028,19.54,1.55


In [27]:
product_page_views_conversion = (
    session_model
    .groupby("product_page_views")
    .agg(
        sessions=("session", "count"),
        orders=("generated_order", "sum"),
        conversion_rate=("generated_order", "mean")
    )
    .reset_index()
)

product_page_views_conversion["share_sessions"] = (
    product_page_views_conversion["sessions"]
    / product_page_views_conversion["sessions"].sum()
    * 100
).round(2)

product_page_views_conversion["conversion_rate"] = (
    product_page_views_conversion["conversion_rate"]
    * 100
).round(2)

product_page_views_conversion = product_page_views_conversion.sort_values(
    "product_page_views"
)

product_page_views_conversion

,product_page_views,sessions,orders,conversion_rate,share_sessions
0,0,177670,2583,1.45,52.19
1,1,116911,1700,1.45,34.34
2,2,24016,1060,4.41,7.05
3,3,9676,675,6.98,2.84
4,4,4663,495,10.62,1.37
5,5,2621,299,11.41,0.77
6,6,1415,191,13.50,0.42
7,7,1002,146,14.57,0.29
8,8,629,101,16.06,0.18
9,9,422,75,17.77,0.12


### Customer Model

In [ ]:
df_user = df.copy()

# Datetime + ordenação
df_user["event_date"] = pd.to_datetime(df_user["event_date"])
df_user = df_user.sort_values(["user", "event_date"]).reset_index(drop=True)

# Flags auxiliares
df_user["is_page_view"] = (df_user["event_type"] == "page_view").astype(int)
df_user["is_add_to_cart"] = (df_user["event_type"] == "add_to_cart").astype(int)
df_user["is_order"] = (df_user["event_type"] == "order").astype(int)
df_user["is_product_page_view"] = (
    (df_user["page_type"] == "product_page") &
    (df_user["event_type"] == "page_view")
).astype(int)

# Para contar page_types apenas quando houve page_view
df_user["page_type_if_pageview"] = df_user["page_type"].where(
    df_user["event_type"] == "page_view"
)

# Ordem dos eventos dentro do usuário
df_user["event_order"] = df_user.groupby("user").cumcount() + 1

# Tabela base por user
user_model = (
    df_user.groupby("user", sort=False)
      .agg(
          first_event_datetime=("event_date", "min"),
          last_event_datetime=("event_date", "max"),

          number_sessions=("session", "nunique"),

          generated_order=("is_order", "max"),

          number_pageviews=("is_page_view", "sum"),
          has_add_to_cart=("is_add_to_cart", "max"),
          number_add_to_cart=("is_add_to_cart", "sum"),

          different_page_types_viewed=("page_type_if_pageview", "nunique"),
          product_page_views=("is_product_page_view", "sum")
      )
      .reset_index()
)

# Primeira página do usuário, ignorando order_page
first_page_user = (
    df_user[df_user["page_type"] != "order_page"]
      .sort_values(["user", "event_date"])
      .groupby("user", sort=False)
      .first()[["page_type"]]
      .rename(columns={"page_type": "first_page"})
      .reset_index()
)

# Primeiro add_to_cart por usuário
first_cart_user = (
    df_user[df_user["is_add_to_cart"] == 1]
      .groupby("user", sort=False)["event_order"]
      .min()
      .reset_index(name="first_add_to_cart_position")
)

user_model = user_model.merge(first_cart_user, on="user", how="left")
user_model = user_model.merge(first_page_user, on="user", how="left")

# Pageviews até o primeiro add_to_cart
df_user_with_cart = df_user.merge(first_cart_user, on="user", how="left")

pageviews_before_cart_user = (
    df_user_with_cart[
        (df_user_with_cart["is_page_view"] == 1) &
        (df_user_with_cart["event_order"] < df_user_with_cart["first_add_to_cart_position"])
    ]
    .groupby("user", sort=False)
    .size()
    .reset_index(name="pageviews_before_first_add_to_cart")
)

user_model = user_model.merge(pageviews_before_cart_user, on="user", how="left")

# Para usuários sem add_to_cart, preencher com total de pageviews
user_model["pageviews_before_first_add_to_cart"] = (
    user_model["pageviews_before_first_add_to_cart"]
    .fillna(user_model["number_pageviews"])
)

user_model["first_add_to_cart_position"] = (
    user_model["first_add_to_cart_position"].fillna(0)
)

# Add_to_cart repetido do mesmo produto dentro do usuário
cart_user_df = df_user[
    (df_user["is_add_to_cart"] == 1) &
    (df_user["product"].notna()) &
    (df_user["product"] != 0)
].copy()

product_cart_by_user = (
    cart_user_df.groupby(["user", "product"], sort=False)
      .size()
      .reset_index(name="same_product_cart_count")
)

repeated_product_user_features = (
    product_cart_by_user
      .assign(is_repeated_product=lambda x: (x["same_product_cart_count"] > 1).astype(int))
      .groupby("user", sort=False)
      .agg(
          max_same_product_add_to_cart=("same_product_cart_count", "max"),
          repeated_products_count=("is_repeated_product", "sum"),
          has_repeated_product_add_to_cart=("is_repeated_product", "max")
      )
      .reset_index()
)

user_model = user_model.merge(repeated_product_user_features, on="user", how="left")

user_model[[
    "max_same_product_add_to_cart",
    "repeated_products_count",
    "has_repeated_product_add_to_cart"
]] = user_model[[
    "max_same_product_add_to_cart",
    "repeated_products_count",
    "has_repeated_product_add_to_cart"
]].fillna(0)

user_model.head()

### Behavior tests

In [9]:
# Todas as transições cujo próximo passo foi Order
before_order = (
    df[df["next_page"] == "order_page"]
    .groupby("page_type")
    .size()
    .reset_index(name="sessions")
    .sort_values("sessions", ascending=False)
)

before_order["pct"] = (
    before_order["sessions"] /
    before_order["sessions"].sum()
)

before_order["pct"] = (
    before_order["pct"] * 100
).round(1)

before_order

,page_type,sessions,pct
2,product_page,4051,55.2
0,listing_page,1427,19.4
3,search_listing_page,963,13.1
1,order_page,901,12.3


In [12]:
import pandas as pd
import numpy as np

# ==========================================================
# PREPARAÇÃO
# ==========================================================

df = df.copy()

df["event_date"] = pd.to_datetime(df["event_date"])
df = df.sort_values(["session", "event_date"])

# Conversão da sessão
df["is_order"] = (df["page_type"] == "order_page").astype(int)

# Próximos eventos
df["next_page"] = df.groupby("session")["page_type"].shift(-1)
df["next_time"] = df.groupby("session")["event_date"].shift(-1)

# Mudança de página
df["page_changed"] = (
    (df["page_type"] != df["next_page"])
    & df["next_page"].notna()
)

# Tempo até próximo evento
df["seconds_to_next"] = (
    df["next_time"] - df["event_date"]
).dt.total_seconds()

# Página anterior
df["prev_page"] = df.groupby("session")["page_type"].shift(1)

# ==========================================================
# FEATURES DA SESSÃO
# ==========================================================

session_features = (
    df.groupby("session")
      .agg(
          user=("user","first"),

          # TARGET
          buyer=("is_order","max"),

          # Volume
          total_events=("event_type","count"),
          pageviews=("event_type", lambda x: (x=="page_view").sum()),

          # Navegação
          unique_pages=("page_type","nunique"),
          page_changes=("page_changed","sum"),

          # Entrada / saída
          first_page=("page_type","first"),
          last_page=("page_type","last"),

          # Tempo
          start_time=("event_date","min"),
          end_time=("event_date","max")
      )
      .reset_index()
)

session_features["session_duration_sec"] = (
    session_features["end_time"] -
    session_features["start_time"]
).dt.total_seconds()

# ==========================================================
# FLAGS DE PÁGINAS VISITADAS
# ==========================================================

pages = [
    "search_listing_page",
    "listing_page",
    "product_page",
    "order_page"
]

for page in pages:

    visited = (
        df.groupby("session")["page_type"]
          .apply(lambda x: (x == page).any())
          .rename(f"visited_{page}")
    )

    session_features = session_features.merge(
        visited,
        on="session",
        how="left"
    )

# ==========================================================
# QUANTAS VEZES VISITOU PRODUCT PAGE
# ==========================================================

product_visits = (
    df[df["page_type"]=="product_page"]
    .groupby("session")
    .size()
    .rename("product_page_visits")
)

session_features = session_features.merge(
    product_visits,
    on="session",
    how="left"
)

session_features["product_page_visits"] = (
    session_features["product_page_visits"]
    .fillna(0)
)

# ==========================================================
# BOUNCE
# ==========================================================

session_features["is_bounce"] = (
    session_features["total_events"] == 1
)

# ==========================================================
# LOOPS (A -> B -> A)
# ==========================================================

df["is_loop"] = (
    (df["prev_page"] == df["next_page"])
    & (df["page_type"] != df["prev_page"])
)

loops = (
    df.groupby("session")["is_loop"]
      .sum()
      .rename("navigation_loops")
)

session_features = session_features.merge(
    loops,
    on="session",
    how="left"
)

session_features["navigation_loops"] = (
    session_features["navigation_loops"]
    .fillna(0)
)

# ==========================================================
# TEMPO ATÉ PRIMEIRA PRODUCT PAGE
# ==========================================================

first_product = (
    df[df["page_type"]=="product_page"]
    .groupby("session")["event_date"]
    .min()
    .rename("first_product_time")
)

session_features = session_features.merge(
    first_product,
    on="session",
    how="left"
)

session_features["time_to_first_product_sec"] = (
    session_features["first_product_time"]
    - session_features["start_time"]
).dt.total_seconds()

# ==========================================================
# TEMPO MÉDIO ENTRE EVENTOS
# ==========================================================

avg_time = (
    df.groupby("session")["seconds_to_next"]
      .mean()
      .rename("avg_seconds_between_events")
)

session_features = session_features.merge(
    avg_time,
    on="session",
    how="left"
)

# ==========================================================
# TRATAMENTO DOS NaN
# ==========================================================

session_features["product_page_visits"] = (
    session_features["product_page_visits"].fillna(0)
)

session_features["navigation_loops"] = (
    session_features["navigation_loops"].fillna(0)
)

session_features["time_to_first_product_sec"] = (
    session_features["time_to_first_product_sec"].fillna(-1)
)

session_features["avg_seconds_between_events"] = (
    session_features["avg_seconds_between_events"].fillna(0)
)

In [13]:
comparison = (
    session_features
    .groupby("buyer")
    .agg(
        sessions=("session","count"),
        avg_events=("total_events","mean"),
        avg_pageviews=("pageviews","mean"),
        avg_unique_pages=("unique_pages","mean"),
        avg_page_changes=("page_changes","mean"),
        avg_duration_sec=("session_duration_sec","mean"),
        avg_product_visits=("product_page_visits","mean"),
        avg_navigation_loops=("navigation_loops","mean"),
        avg_time_to_product=("time_to_first_product_sec","mean"),
        avg_time_between_events=("avg_seconds_between_events","mean"),
        bounce_rate=("is_bounce","mean"),
        pct_search=("visited_search_listing_page","mean"),
        pct_listing=("visited_listing_page","mean"),
        pct_product=("visited_product_page","mean"),
        pct_order=("visited_order_page","mean")
    )
)

# Converter percentuais
for col in [
    "bounce_rate",
    "pct_search",
    "pct_listing",
    "pct_product",
    "pct_order"
]:
    comparison[col] *= 100

comparison = comparison.round(2)

# Deixar no formato desejado
comparison = comparison.T
comparison.columns = ["Non Buyer", "Buyer"]

comparison

,Non Buyer,Buyer
sessions,332806.00,7637.00
avg_events,1.76,6.77
avg_pageviews,1.73,4.75
avg_unique_pages,1.10,2.46
avg_page_changes,0.19,2.31
avg_duration_sec,97.28,910.66
avg_product_visits,0.78,2.99
avg_navigation_loops,0.06,0.43
avg_time_to_product,11.77,97.27
avg_time_between_events,40.22,187.74


In [25]:
import pandas as pd
import numpy as np

# ==========================================================
# 1. PREPARAÇÃO
# ==========================================================

df = df.copy()

df["event_date"] = pd.to_datetime(df["event_date"])

df = (
    df.sort_values(["session", "event_date"])
      .reset_index(drop=True)
)

# ==========================================================
# 2. FLAGS AUXILIARES
# ==========================================================

df["is_order"] = df["page_type"].eq("order_page")
df["is_pageview"] = df["event_type"].eq("page_view")

df["is_product_page"] = df["page_type"].eq("product_page")
df["is_search_page"] = df["page_type"].eq("search_listing_page")
df["is_listing_page"] = df["page_type"].eq("listing_page")

# Add to Cart:
# product == 0 -> NÃO fez add_to_cart
# product > 0  -> fez add_to_cart
df["is_add_to_cart"] = df["product"].ne(0)

# ==========================================================
# 3. FEATURES BASE DA SESSÃO
# ==========================================================

session_features = (
    df.groupby("session")
      .agg(
          user=("user", "first"),
          buyer=("is_order", "max"),

          first_page=("page_type", "first"),
          last_page=("page_type", "last"),

          start_time=("event_date", "min"),
          end_time=("event_date", "max")
      )
      .reset_index()
)

# ==========================================================
# TOTAL DE EVENTOS
# ==========================================================

total_events = (
    df.groupby("session")
      .size()
      .rename("total_events")
      .reset_index()
)

session_features = session_features.merge(
    total_events,
    on="session",
    how="left"
)

# ==========================================================
# PAGEVIEWS
# ==========================================================

pageviews = (
    df.groupby("session")["is_pageview"]
      .sum()
      .rename("pageviews")
      .reset_index()
)

session_features = session_features.merge(
    pageviews,
    on="session",
    how="left"
)

# ==========================================================
# UNIQUE PAGES
# ==========================================================

unique_pages = (
    df.groupby("session")["page_type"]
      .nunique()
      .rename("unique_pages")
      .reset_index()
)

session_features = session_features.merge(
    unique_pages,
    on="session",
    how="left"
)

# ==========================================================
# PAGE CHANGES
# ==========================================================

df["next_page"] = df.groupby("session")["page_type"].shift(-1)

df["page_changed"] = (
    df["next_page"].notna()
    &
    (df["page_type"] != df["next_page"])
)

page_changes = (
    df.groupby("session")["page_changed"]
      .sum()
      .rename("page_changes")
      .reset_index()
)

session_features = session_features.merge(
    page_changes,
    on="session",
    how="left"
)

# ==========================================================
# SESSION DURATION
# ==========================================================

session_features["session_duration_sec"] = (
    session_features["end_time"]
    - session_features["start_time"]
).dt.total_seconds()

# ==========================================================
# TEMPO ENTRE EVENTOS
# ==========================================================

df["next_time"] = df.groupby("session")["event_date"].shift(-1)

df["seconds_to_next"] = (
    df["next_time"]
    - df["event_date"]
).dt.total_seconds()

avg_seconds = (
    df.groupby("session")["seconds_to_next"]
      .mean()
      .rename("avg_seconds_between_events")
      .reset_index()
)

session_features = session_features.merge(
    avg_seconds,
    on="session",
    how="left"
)

# ==========================================================
# TEMPO ENTRE PRIMEIRO E SEGUNDO EVENTO
# Mantém NaN para sessões com apenas 1 evento
# ==========================================================

df["event_rank"] = df.groupby("session").cumcount() + 1

first_event_time = (
    df[df["event_rank"] == 1]
      .groupby("session")["event_date"]
      .min()
      .rename("first_event_time")
      .reset_index()
)

second_event_time = (
    df[df["event_rank"] == 2]
      .groupby("session")["event_date"]
      .min()
      .rename("second_event_time")
      .reset_index()
)

time_to_second_event = (
    first_event_time
    .merge(second_event_time, on="session", how="left")
)

time_to_second_event["time_to_second_event_sec"] = (
    time_to_second_event["second_event_time"]
    - time_to_second_event["first_event_time"]
).dt.total_seconds()

session_features = session_features.merge(
    time_to_second_event[["session", "time_to_second_event_sec"]],
    on="session",
    how="left"
)

# ==========================================================
# BOUNCE
# ==========================================================

session_features["is_bounce"] = (
    session_features["total_events"] == 1
)

# ==========================================================
# PRIMEIRA PÁGINA
# ==========================================================

session_features["started_search"] = (
    session_features["first_page"] == "search_listing_page"
)

session_features["started_listing"] = (
    session_features["first_page"] == "listing_page"
)

session_features["started_product"] = (
    session_features["first_page"] == "product_page"
)

# ==========================================================
# NULOS
# ==========================================================

fill_zero = [
    "pageviews",
    "unique_pages",
    "page_changes",
    "avg_seconds_between_events"
]

session_features[fill_zero] = (
    session_features[fill_zero]
    .fillna(0)
)

session_features.head()

,session,user,buyer,first_page,last_page,start_time,end_time,total_events,pageviews,unique_pages,page_changes,session_duration_sec,avg_seconds_between_events,time_to_second_event_sec,is_bounce,started_search,started_listing,started_product
0,10000000052138714153s,8035194775607769271u,False,search_listing_page,search_listing_page,2022-10-12 16:26:46,2022-10-12 16:26:46,1,1,1,0,0.0,0.0,NaN,True,True,False,False
1,10000003251787879077s,2613381957342920411u,False,search_listing_page,search_listing_page,2022-10-01 19:16:18,2022-10-01 19:16:56,2,2,1,0,38.0,38.0,38.0,False,True,False,False
2,10000028460721229362s,12451948957440700749u,False,listing_page,listing_page,2022-10-13 09:08:27,2022-10-13 09:08:33,2,2,1,0,6.0,6.0,6.0,False,False,True,False
3,10000145746398845278s,16615957149200537975u,False,listing_page,listing_page,2022-10-04 12:59:05,2022-10-04 12:59:05,1,1,1,0,0.0,0.0,NaN,True,False,True,False
4,1000015740551295390s,6456170714504168075u,False,listing_page,listing_page,2022-10-03 16:03:53,2022-10-03 16:03:53,1,1,1,0,0.0,0.0,NaN,True,False,True,False


In [26]:
# ==========================================================
# BLOCO 2 - FEATURES DE NAVEGAÇÃO
# ==========================================================

# Remove colunas antigas se você estiver rerodando a célula
cols_to_drop = [
    "visited_search_listing_page",
    "visited_listing_page",
    "visited_product_page",
    "visited_order_page",
    "product_page_visits",
    "navigation_loops",
    "returned_to_search"
]

session_features = session_features.drop(
    columns=[c for c in cols_to_drop if c in session_features.columns]
)

# ==========================================================
# 1. VISITOU CADA PÁGINA
# ==========================================================

visited_pages = (
    df.groupby("session")
      .agg(
          visited_search_listing_page=("is_search_page", "max"),
          visited_listing_page=("is_listing_page", "max"),
          visited_product_page=("is_product_page", "max"),
          visited_order_page=("is_order", "max")
      )
      .reset_index()
)

session_features = session_features.merge(
    visited_pages,
    on="session",
    how="left"
)

# ==========================================================
# 2. QUANTIDADE DE VISITAS À PRODUCT PAGE
# ==========================================================

product_visits = (
    df.groupby("session")["is_product_page"]
      .sum()
      .rename("product_page_visits")
      .reset_index()
)

session_features = session_features.merge(
    product_visits,
    on="session",
    how="left"
)

# ==========================================================
# 3. LOOPS A -> B -> A
# ==========================================================

df["prev_page"] = df.groupby("session")["page_type"].shift(1)
df["next_page"] = df.groupby("session")["page_type"].shift(-1)

df["is_loop"] = (
    (df["prev_page"] == df["next_page"]) &
    (df["page_type"] != df["prev_page"])
)

loops = (
    df.groupby("session")["is_loop"]
      .sum()
      .rename("navigation_loops")
      .reset_index()
)

session_features = session_features.merge(
    loops,
    on="session",
    how="left"
)

# ==========================================================
# 4. RETORNOU À SEARCH APÓS VISITAR PRODUCT
# ==========================================================

df["event_order"] = df.groupby("session").cumcount()

first_product_order = (
    df[df["is_product_page"]]
      .groupby("session")["event_order"]
      .min()
      .rename("first_product_order")
      .reset_index()
)

df = df.drop(
    columns=["first_product_order"],
    errors="ignore"
)

df = df.merge(
    first_product_order,
    on="session",
    how="left"
)

returned_to_search = (
    (
        df["is_search_page"] &
        (df["event_order"] > df["first_product_order"])
    )
    .groupby(df["session"])
    .max()
    .rename("returned_to_search")
    .reset_index()
)

session_features = session_features.merge(
    returned_to_search,
    on="session",
    how="left"
)

# ==========================================================
# 5. NULOS
# ==========================================================

fill_false = [
    "visited_search_listing_page",
    "visited_listing_page",
    "visited_product_page",
    "visited_order_page",
    "returned_to_search"
]

session_features[fill_false] = (
    session_features[fill_false]
    .fillna(False)
)

fill_zero = [
    "product_page_visits",
    "navigation_loops"
]

session_features[fill_zero] = (
    session_features[fill_zero]
    .fillna(0)
)

session_features.head()

,session,user,buyer,first_page,last_page,start_time,end_time,total_events,pageviews,unique_pages,...,started_search,started_listing,started_product,visited_search_listing_page,visited_listing_page,visited_product_page,visited_order_page,product_page_visits,navigation_loops,returned_to_search
0,10000000052138714153s,8035194775607769271u,False,search_listing_page,search_listing_page,2022-10-12 16:26:46,2022-10-12 16:26:46,1,1,1,...,True,False,False,True,False,False,False,0,0,False
1,10000003251787879077s,2613381957342920411u,False,search_listing_page,search_listing_page,2022-10-01 19:16:18,2022-10-01 19:16:56,2,2,1,...,True,False,False,True,False,False,False,0,0,False
2,10000028460721229362s,12451948957440700749u,False,listing_page,listing_page,2022-10-13 09:08:27,2022-10-13 09:08:33,2,2,1,...,False,True,False,False,True,False,False,0,0,False
3,10000145746398845278s,16615957149200537975u,False,listing_page,listing_page,2022-10-04 12:59:05,2022-10-04 12:59:05,1,1,1,...,False,True,False,False,True,False,False,0,0,False
4,1000015740551295390s,6456170714504168075u,False,listing_page,listing_page,2022-10-03 16:03:53,2022-10-03 16:03:53,1,1,1,...,False,True,False,False,True,False,False,0,0,False


In [27]:
# ==========================================================
# BLOCO 3 - FEATURES DE PRODUTO
# ==========================================================

# ==========================================================
# 1. FEZ ADD TO CART?
# ==========================================================

added_to_cart = (
    df.groupby("session")["is_add_to_cart"]
      .max()
      .rename("added_to_cart")
      .reset_index()
)

session_features = session_features.merge(
    added_to_cart,
    on="session",
    how="left"
)

# ==========================================================
# 2. QUANTIDADE DE ADD TO CART
# ==========================================================

add_to_cart_count = (
    df.groupby("session")["is_add_to_cart"]
      .sum()
      .rename("add_to_cart_count")
      .reset_index()
)

session_features = session_features.merge(
    add_to_cart_count,
    on="session",
    how="left"
)

# ==========================================================
# 3. TEMPO ATÉ PRIMEIRA PRODUCT PAGE
# ==========================================================

first_product_time = (
    df[df["is_product_page"]]
      .groupby("session")["event_date"]
      .min()
      .rename("first_product_time")
      .reset_index()
)

session_features = session_features.merge(
    first_product_time,
    on="session",
    how="left"
)

session_features["time_to_first_product_sec"] = (
    session_features["first_product_time"]
    - session_features["start_time"]
).dt.total_seconds()

# ==========================================================
# 4. TEMPO ATÉ PRIMEIRO ADD TO CART
# ==========================================================

first_add_to_cart = (
    df[df["is_add_to_cart"]]
      .groupby("session")["event_date"]
      .min()
      .rename("first_add_to_cart_time")
      .reset_index()
)

session_features = session_features.merge(
    first_add_to_cart,
    on="session",
    how="left"
)

session_features["time_to_first_add_to_cart_sec"] = (
    session_features["first_add_to_cart_time"]
    - session_features["start_time"]
).dt.total_seconds()

# ==========================================================
# 5. PRODUCT VIEWS ANTES DO PRIMEIRO ADD TO CART
# ==========================================================

product_before_add = (
    df.merge(
        first_add_to_cart,
        on="session",
        how="left"
    )
)

product_before_add = product_before_add[
    product_before_add["is_product_page"]
]

product_before_add = product_before_add[
    product_before_add["event_date"]
    < product_before_add["first_add_to_cart_time"]
]

product_views_before_add = (
    product_before_add
      .groupby("session")
      .size()
      .rename("product_views_before_add_to_cart")
      .reset_index()
)

session_features = session_features.merge(
    product_views_before_add,
    on="session",
    how="left"
)

# ==========================================================
# 6. NULOS
# ==========================================================

fill_false = [
    "added_to_cart"
]

for col in fill_false:
    session_features[col] = (
        session_features[col]
        .fillna(False)
    )

fill_zero = [
    "add_to_cart_count",
    "product_views_before_add_to_cart"
]

for col in fill_zero:
    session_features[col] = (
        session_features[col]
        .fillna(0)
    )

fill_minus_one = [
    "time_to_first_product_sec",
    "time_to_first_add_to_cart_sec"
]

for col in fill_minus_one:
    session_features[col] = (
        session_features[col]
        .fillna(-1)
    )

session_features.head()

,session,user,buyer,first_page,last_page,start_time,end_time,total_events,pageviews,unique_pages,...,product_page_visits,navigation_loops,returned_to_search,added_to_cart,add_to_cart_count,first_product_time,time_to_first_product_sec,first_add_to_cart_time,time_to_first_add_to_cart_sec,product_views_before_add_to_cart
0,10000000052138714153s,8035194775607769271u,False,search_listing_page,search_listing_page,2022-10-12 16:26:46,2022-10-12 16:26:46,1,1,1,...,0,0,False,False,0,NaT,-1.0,NaT,-1.0,0.0
1,10000003251787879077s,2613381957342920411u,False,search_listing_page,search_listing_page,2022-10-01 19:16:18,2022-10-01 19:16:56,2,2,1,...,0,0,False,False,0,NaT,-1.0,NaT,-1.0,0.0
2,10000028460721229362s,12451948957440700749u,False,listing_page,listing_page,2022-10-13 09:08:27,2022-10-13 09:08:33,2,2,1,...,0,0,False,False,0,NaT,-1.0,NaT,-1.0,0.0
3,10000145746398845278s,16615957149200537975u,False,listing_page,listing_page,2022-10-04 12:59:05,2022-10-04 12:59:05,1,1,1,...,0,0,False,False,0,NaT,-1.0,NaT,-1.0,0.0
4,1000015740551295390s,6456170714504168075u,False,listing_page,listing_page,2022-10-03 16:03:53,2022-10-03 16:03:53,1,1,1,...,0,0,False,False,0,NaT,-1.0,NaT,-1.0,0.0


In [28]:
# ==========================================================
# BLOCO 4 - COMPARISON BUYER VS NON BUYER
# ==========================================================

comparison = (
    session_features
    .groupby("buyer")
    .agg(
        sessions=("session", "count"),

        # Engagement
        avg_events=("total_events", "mean"),
        avg_pageviews=("pageviews", "mean"),
        avg_unique_pages=("unique_pages", "mean"),
        avg_page_changes=("page_changes", "mean"),
        avg_duration_sec=("session_duration_sec", "mean"),
        avg_time_between_events=("avg_seconds_between_events", "mean"),
        avg_time_to_second_event=("time_to_second_event_sec", "mean"),

        # Navigation
        avg_product_visits=("product_page_visits", "mean"),
        avg_navigation_loops=("navigation_loops", "mean"),

        # Product Discovery
        avg_time_to_product=("time_to_first_product_sec", "mean"),
        avg_time_to_add_to_cart=("time_to_first_add_to_cart_sec", "mean"),
        avg_product_views_before_add=("product_views_before_add_to_cart", "mean"),

        # Bounce
        bounce_rate=("is_bounce", "mean"),

        # Visited Pages
        pct_search=("visited_search_listing_page", "mean"),
        pct_listing=("visited_listing_page", "mean"),
        pct_product=("visited_product_page", "mean"),

        # Entry Point
        pct_started_search=("started_search", "mean"),
        pct_started_listing=("started_listing", "mean"),
        pct_started_product=("started_product", "mean"),

        # Add to Cart
        pct_add_to_cart=("added_to_cart", "mean"),
        avg_add_to_cart=("add_to_cart_count", "mean"),

        # Comparison Behaviour
        pct_returned_to_search=("returned_to_search", "mean")
    )
)

# ==========================================================
# CONVERTER PERCENTUAIS
# ==========================================================

pct_columns = [
    "bounce_rate",
    "pct_search",
    "pct_listing",
    "pct_product",
    "pct_started_search",
    "pct_started_listing",
    "pct_started_product",
    "pct_add_to_cart",
    "pct_returned_to_search"
]

comparison[pct_columns] = comparison[pct_columns] * 100

# ==========================================================
# FORMATAÇÃO
# ==========================================================

comparison = comparison.round(2)

comparison = comparison.T

comparison.columns = [
    "Non Buyer",
    "Buyer"
]

comparison

,Non Buyer,Buyer
sessions,332806.00,7637.00
avg_events,1.76,6.77
avg_pageviews,1.73,4.75
avg_unique_pages,1.10,2.46
avg_page_changes,0.19,2.31
avg_duration_sec,97.28,910.66
avg_time_between_events,40.22,187.74
avg_time_to_second_event,164.37,221.45
avg_product_visits,0.78,2.99
avg_navigation_loops,0.06,0.43


In [18]:
df["is_add_to_cart"].value_counts(dropna=False)

is_add_to_cart
True    637238
Name: count, dtype: int64

In [29]:
session_features.groupby("buyer")["time_to_second_event_sec"].describe()

,count,mean,std,min,25%,50%,75%,max
buyer,,,,,,,,
False,83515.0,164.367694,328.537912,0.0,12.0,45.0,143.0,8908.0
True,6694.0,221.450105,393.480414,0.0,27.0,78.0,235.0,6748.0


In [30]:
# ==========================================================
# FUNIL TRADICIONAL - SESSION LEVEL
# ==========================================================

funnel = pd.DataFrame({
    "stage_order": [1, 2, 3, 4, 5],
    "stage": [
        "Website Visit",
        "Search / Listing",
        "Product",
        "Add to Cart",
        "Order"
    ],
    "sessions": [
        session_features["session"].nunique(),
        session_features[
            session_features["visited_search_listing_page"] |
            session_features["visited_listing_page"]
        ]["session"].nunique(),
        session_features[
            session_features["visited_product_page"]
        ]["session"].nunique(),
        session_features[
            session_features["added_to_cart"]
        ]["session"].nunique(),
        session_features[
            session_features["buyer"]
        ]["session"].nunique()
    ]
})

# Conversão em relação à etapa anterior
funnel["conversion_from_previous_stage"] = (
    funnel["sessions"] / funnel["sessions"].shift(1)
)

# Drop-off em relação à etapa anterior
funnel["dropoff_from_previous_stage"] = (
    1 - funnel["conversion_from_previous_stage"]
)

# Conversão acumulada desde Website Visit
funnel["conversion_from_visit"] = (
    funnel["sessions"] / funnel.loc[0, "sessions"]
)

# Formatação percentual
funnel["conversion_from_previous_stage_pct"] = (
    funnel["conversion_from_previous_stage"] * 100
).round(2)

funnel["dropoff_from_previous_stage_pct"] = (
    funnel["dropoff_from_previous_stage"] * 100
).round(2)

funnel["conversion_from_visit_pct"] = (
    funnel["conversion_from_visit"] * 100
).round(2)

# Ajustar primeira linha
funnel.loc[0, "conversion_from_previous_stage_pct"] = 100
funnel.loc[0, "dropoff_from_previous_stage_pct"] = 0

funnel

,stage_order,stage,sessions,conversion_from_previous_stage,dropoff_from_previous_stage,conversion_from_visit,conversion_from_previous_stage_pct,dropoff_from_previous_stage_pct,conversion_from_visit_pct
0,1,Website Visit,340443,NaN,NaN,1.000000,100.00,0.00,100.00
1,2,Search / Listing,200710,0.589555,0.410445,0.589555,58.96,41.04,58.96
2,3,Product,162856,0.811400,0.188600,0.478365,81.14,18.86,47.84
3,4,Add to Cart,10667,0.065500,0.934500,0.031333,6.55,93.45,3.13
4,5,Order,7637,0.715946,0.284054,0.022433,71.59,28.41,2.24


In [31]:
df.loc[df["product"] > 0, "session"].nunique()

10667

### Abnormal events

#### Outliers events per session

In [7]:
import pandas as pd

# Total de eventos por sessão
events_per_session = (
    df.groupby("session")
    .size()
    .reset_index(name="total_events")
)

# Percentis desejados
percentiles = events_per_session["total_events"].quantile([
    0,
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    0.995,
    0.999,
    1
])

# IQR
q1 = events_per_session["total_events"].quantile(0.25)
q3 = events_per_session["total_events"].quantile(0.75)
iqr = q3 - q1

iqr_1_5_upper = q3 + 1.5 * iqr
iqr_3_upper = q3 + 3 * iqr

# Tabela final
events_distribution = pd.DataFrame({
    "metric": [
        "min",
        "p1",
        "p5",
        "p10",
        "p25",
        "p50",
        "p75",
        "p90",
        "p95",
        "p99",
        "p99.5",
        "p99.9",
        "max",
        "IQR",
        "IQR upper 1.5x",
        "IQR upper 3x"
    ],
    "value": [
        percentiles.loc[0],
        percentiles.loc[0.01],
        percentiles.loc[0.05],
        percentiles.loc[0.10],
        percentiles.loc[0.25],
        percentiles.loc[0.50],
        percentiles.loc[0.75],
        percentiles.loc[0.90],
        percentiles.loc[0.95],
        percentiles.loc[0.99],
        percentiles.loc[0.995],
        percentiles.loc[0.999],
        percentiles.loc[1],
        iqr,
        iqr_1_5_upper,
        iqr_3_upper
    ]
})

events_distribution

,metric,value
0,min,1.0
1,p1,1.0
2,p5,1.0
3,p10,1.0
4,p25,1.0
5,p50,1.0
6,p75,2.0
7,p90,3.0
8,p95,5.0
9,p99,14.0


In [9]:
# Quantidade de eventos por sessão
events_per_session = (
    df.groupby("session")
    .size()
    .reset_index(name="total_events")
    .sort_values("total_events", ascending=False)
)

events_per_session.head(30)

,session,total_events
331081,9543669642136985868s,1264
99131,14816963559813395593s,346
207526,3509679068011838002s,251
282499,7160321411091116049s,203
266980,6410548667176451624s,193
39007,11887676363519834043s,187
112383,1546695377287335747s,175
8250,10397303304025085330s,170
320579,9034927356284785294s,143
224693,4347471093706395840s,131


In [10]:
session_id = "9543669642136985868s"

df[
    df["session"] == session_id
].sort_values("event_date")

,event_date,session,user,page_type,event_type,product,is_order,is_pageview,is_product_page,is_search_page,...,is_add_to_cart,next_page,prev_page,next_time,page_changed,seconds_to_next,is_loop,event_order,page_type_exploration,first_product_order
618566,2022-10-13 13:52:19,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,NaN,2022-10-13 13:52:20,False,1.0,False,1,listing_page,NaN
618567,2022-10-13 13:52:20,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 13:52:21,False,1.0,False,2,listing_page,NaN
618568,2022-10-13 13:52:21,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 13:52:26,False,5.0,False,3,listing_page,NaN
618569,2022-10-13 13:52:26,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 13:52:38,False,12.0,False,4,listing_page,NaN
618570,2022-10-13 13:52:38,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 13:52:42,False,4.0,False,5,listing_page,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619825,2022-10-13 15:06:29,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 15:06:33,False,4.0,False,1260,listing_page,NaN
619826,2022-10-13 15:06:33,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 15:06:38,False,5.0,False,1261,listing_page,NaN
619827,2022-10-13 15:06:38,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 15:06:42,False,4.0,False,1262,listing_page,NaN
619828,2022-10-13 15:06:42,9543669642136985868s,11769744300065907078u,listing_page,page_view,0,False,True,False,False,...,False,listing_page,listing_page,2022-10-13 15:06:47,False,5.0,False,1263,listing_page,NaN


In [11]:
events_per_session = (
    df.groupby("session")
    .size()
    .reset_index(name="total_events")
    .sort_values("total_events", ascending=False)
    .reset_index(drop=True)
)

p99 = events_per_session["total_events"].quantile(0.99)

events_per_session["next_total_events"] = events_per_session["total_events"].shift(-1)

events_per_session["gap_vs_next"] = (
    events_per_session["total_events"] /
    events_per_session["next_total_events"] - 1
)

extreme_gap_outliers = events_per_session[
    (events_per_session["total_events"] > p99) &
    (events_per_session["gap_vs_next"] >= 1)
]

extreme_gap_outliers

,session,total_events,next_total_events,gap_vs_next
0,9543669642136985868s,1264,346.0,2.653179


#### Avg time between events

In [12]:
# Ordenar eventos
df_time = (
    df.sort_values(["session", "event_date"])
    .copy()
)

# Próximo evento dentro da mesma sessão
df_time["next_event_time"] = (
    df_time.groupby("session")["event_date"]
    .shift(-1)
)

# Tempo até o próximo evento (segundos)
df_time["time_to_next_event_sec"] = (
    df_time["next_event_time"] - df_time["event_date"]
).dt.total_seconds()

# Remover último evento de cada sessão
df_time = df_time[
    df_time["time_to_next_event_sec"].notna()
]

# Média do tempo entre eventos por sessão
avg_time_per_session = (
    df_time.groupby("session")
    .agg(
        avg_time_between_events_sec=(
            "time_to_next_event_sec",
            "mean"
        )
    )
    .reset_index()
)

# Percentis
time_distribution = (
    avg_time_per_session["avg_time_between_events_sec"]
    .quantile([
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ])
    .reset_index()
)

time_distribution.columns = ["percentile", "seconds"]

time_distribution

,percentile,seconds
0,0.01,2.6
1,0.05,5.0
2,0.10,7.0
3,0.25,22.5
4,0.50,63.3
5,0.75,167.0
6,0.90,422.0
7,0.95,724.5
8,0.99,1463.0


In [13]:
import pandas as pd

df["event_date"] = pd.to_datetime(df["event_date"])

df_time = (
    df.sort_values(["session", "event_date"])
    .copy()
)

# Próximo evento dentro da sessão
df_time["next_event_time"] = (
    df_time.groupby("session")["event_date"].shift(-1)
)

# Tempo até o próximo evento
df_time["time_to_next_event_sec"] = (
    df_time["next_event_time"] - df_time["event_date"]
).dt.total_seconds()

# Remove sessões/eventos sem próximo evento
df_time = df_time[df_time["time_to_next_event_sec"].notna()].copy()

# Média de tempo por sessão
avg_time_per_session = (
    df_time.groupby("session", as_index=False)
    .agg(
        avg_time_between_events_sec=("time_to_next_event_sec", "mean")
    )
)

# Total de eventos por sessão
events_per_session = (
    df.groupby("session")
    .size()
    .reset_index(name="total_events")
)

avg_time_per_session = avg_time_per_session.merge(
    events_per_session,
    on="session",
    how="left"
)

# Buckets por percentis
avg_time_per_session["percentile_bucket"] = pd.qcut(
    avg_time_per_session["avg_time_between_events_sec"],
    q=[0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1],
    labels=[
        "P0-P1",
        "P1-P5",
        "P5-P10",
        "P10-P25",
        "P25-P50",
        "P50-P75",
        "P75-P90",
        "P90-P95",
        "P95-P99",
        "P99-P100"
    ],
    duplicates="drop"
)

summary_time_percentiles = (
    avg_time_per_session
    .groupby("percentile_bucket", observed=True)
    .agg(
        sessions=("session", "count"),
        avg_time_sec=("avg_time_between_events_sec", "mean"),
        median_time_sec=("avg_time_between_events_sec", "median"),
        avg_events=("total_events", "mean"),
        max_events=("total_events", "max")
    )
    .reset_index()
)

summary_time_percentiles

,percentile_bucket,sessions,avg_time_sec,median_time_sec,avg_events,max_events
0,P0-P1,904,1.311227,1.5,2.321903,76
1,P1-P5,4826,4.298708,4.0,2.442603,1264
2,P5-P10,3313,6.394186,6.0,2.376094,29
3,P10-P25,13528,14.140588,14.0,3.768554,93
4,P25-P50,22535,40.909749,40.0,4.848591,346
5,P50-P75,22563,105.171534,100.5,5.186500,251
6,P75-P90,13521,261.973870,247.0,4.582058,69
7,P90-P95,4511,547.479196,533.0,3.332964,24
8,P95-P99,3609,1011.133552,968.0,2.522582,16
9,P99-P100,899,1811.394049,1677.0,2.096774,5


#### Orders without add to cart before

In [18]:
repeated_distribution = (
    repeated_cart
    .groupby("add_to_cart_count_same_product", as_index=False)
    .agg(session_product_pairs=("session", "count"))
)

repeated_distribution["share"] = (
    repeated_distribution["session_product_pairs"] /
    repeated_distribution["session_product_pairs"].sum() * 100
).round(2)

repeated_distribution

,add_to_cart_count_same_product,session_product_pairs,share
0,1,13568,92.91
1,2,855,5.85
2,3,104,0.71
3,4,38,0.26
4,5,13,0.09
5,6,11,0.08
6,7,7,0.05
7,8,1,0.01
8,9,1,0.01
9,10,2,0.01


In [20]:
sessions_with_add_to_cart = (
    df.loc[df["event_type"] == "add_to_cart", "session"]
    .nunique()
)

print("Sessions with add_to_cart:", sessions_with_add_to_cart)

Sessions with add_to_cart: 10667


### Codigo para usuario - Exportar para PowerBI

In [13]:
import pandas as pd
import numpy as np

df_user = df.copy()
df_user["event_date"] = pd.to_datetime(df_user["event_date"])
df_user = df_user.sort_values(["user", "session", "event_date"]).reset_index(drop=True)

# Flags
df_user["is_pageview"] = df_user["event_type"].eq("page_view")
df_user["is_order"] = df_user["event_type"].eq("order")
df_user["is_add_to_cart"] = df_user["event_type"].eq("add_to_cart")
df_user["is_search_page"] = df_user["page_type"].eq("search_listing_page")
df_user["is_listing_page"] = df_user["page_type"].eq("listing_page")
df_user["is_product_page"] = df_user["page_type"].eq("product_page")
df_user["is_product_page_view"] = df_user["is_pageview"] & df_user["is_product_page"]

# Ordem dos eventos por usuário e por sessão
df_user["user_event_order"] = df_user.groupby("user").cumcount() + 1
df_user["session_event_order"] = df_user.groupby(["user", "session"]).cumcount() + 1

# Next / Previous DENTRO da sessão
df_user["next_page"] = df_user.groupby(["user", "session"])["page_type"].shift(-1)
df_user["prev_page"] = df_user.groupby(["user", "session"])["page_type"].shift(1)
df_user["next_time"] = df_user.groupby(["user", "session"])["event_date"].shift(-1)

df_user["page_changed"] = (
    df_user["next_page"].notna() &
    (df_user["page_type"] != df_user["next_page"])
)

df_user["seconds_to_next"] = (
    df_user["next_time"] - df_user["event_date"]
).dt.total_seconds()

df_user["is_loop"] = (
    (df_user["prev_page"] == df_user["next_page"]) &
    (df_user["page_type"] != df_user["prev_page"])
)

# Unique pages sem order_page
df_user["page_type_exploration"] = df_user["page_type"].where(
    df_user["page_type"] != "order_page"
)

# Base principal por usuário
user_features = (
    df_user.groupby("user", as_index=False)
    .agg(
        first_page=("page_type", "first"),
        first_event_time=("event_date", "min"),
        last_event_time=("event_date", "max"),

        total_events=("event_type", "count"),
        pageviews=("is_pageview", "sum"),
        unique_pages=("page_type_exploration", "nunique"),
        page_changes=("page_changed", "sum"),

        visited_search_listing_page=("is_search_page", "max"),
        visited_listing_page=("is_listing_page", "max"),
        visited_product_page=("is_product_page", "max"),

        product_page_visits=("is_product_page", "sum"),
        product_page_views=("is_product_page_view", "sum"),
        navigation_loops=("is_loop", "sum"),

        added_to_cart=("is_add_to_cart", "max"),
        add_to_cart_count=("is_add_to_cart", "sum"),

        order_events=("is_order", "sum"),
        buyer=("is_order", "max")
    )
)

# Top funnel: started page
user_features["started_search"] = user_features["first_page"].eq("search_listing_page").astype(int)
user_features["started_listing"] = user_features["first_page"].eq("listing_page").astype(int)
user_features["started_product"] = user_features["first_page"].eq("product_page").astype(int)

# Funnel flags
user_features["first_visit"] = 1
user_features["exploring"] = (user_features["total_events"] > 1).astype(int)
user_features["product_discovery"] = (
    (user_features["total_events"] > 1) &
    (user_features["product_page_views"] >= 1)
).astype(int)
user_features["purchase_intent"] = (user_features["add_to_cart_count"] >= 1).astype(int)
user_features["purchase"] = (user_features["order_events"] >= 1).astype(int)
user_features["rebuy"] = (user_features["order_events"] >= 2).astype(int)

# ==========================================================
# MÉTRICAS DE TEMPO CALCULADAS DENTRO DE USER + SESSION
# ==========================================================

# Session duration média por usuário
session_duration = (
    df_user.groupby(["user", "session"], as_index=False)
    .agg(
        session_start=("event_date", "min"),
        session_end=("event_date", "max"),
        session_events=("event_type", "count")
    )
)

session_duration["session_duration_sec"] = (
    session_duration["session_end"] - session_duration["session_start"]
).dt.total_seconds()

user_session_features = (
    session_duration.groupby("user", as_index=False)
    .agg(
        sessions=("session", "nunique"),
        avg_session_duration_sec=("session_duration_sec", "mean"),
        avg_events_per_session=("session_events", "mean")
    )
)

user_features = user_features.merge(user_session_features, on="user", how="left")

# Time to second event por sessão -> média por usuário
session_first_event = (
    df_user[df_user["session_event_order"] == 1]
    [["user", "session", "event_date"]]
    .rename(columns={"event_date": "session_first_event_time"})
)

session_second_event = (
    df_user[df_user["session_event_order"] == 2]
    [["user", "session", "event_date"]]
    .rename(columns={"event_date": "session_second_event_time"})
)

session_second_event_features = session_first_event.merge(
    session_second_event,
    on=["user", "session"],
    how="left"
)

session_second_event_features["time_to_second_event_sec"] = (
    session_second_event_features["session_second_event_time"] -
    session_second_event_features["session_first_event_time"]
).dt.total_seconds()

user_time_to_second = (
    session_second_event_features
    .groupby("user", as_index=False)
    .agg(
        time_to_second_event_sec=("time_to_second_event_sec", "mean"),
        second_event_same_session=("time_to_second_event_sec", "count")
    )
)

user_time_to_second["second_event_same_session"] = (
    user_time_to_second["second_event_same_session"] > 0
).astype(int)

user_features = user_features.merge(user_time_to_second, on="user", how="left")

# Avg time between events por sessão -> média por usuário
session_avg_time_between_events = (
    df_user.dropna(subset=["seconds_to_next"])
    .groupby(["user", "session"], as_index=False)
    .agg(
        session_avg_time_between_events=("seconds_to_next", "mean")
    )
)

user_avg_time_between_events = (
    session_avg_time_between_events
    .groupby("user", as_index=False)
    .agg(
        avg_time_between_events=("session_avg_time_between_events", "mean")
    )
)

user_features = user_features.merge(user_avg_time_between_events, on="user", how="left")

# Time to first product por sessão -> média por usuário
session_first_product = (
    df_user[df_user["is_product_page"]]
    .groupby(["user", "session"], as_index=False)
    .agg(first_product_time=("event_date", "min"))
)

session_time_to_product = session_duration.merge(
    session_first_product,
    on=["user", "session"],
    how="left"
)

session_time_to_product["time_to_first_product_sec"] = (
    session_time_to_product["first_product_time"] -
    session_time_to_product["session_start"]
).dt.total_seconds()

user_time_to_product = (
    session_time_to_product
    .groupby("user", as_index=False)
    .agg(
        time_to_first_product_sec=("time_to_first_product_sec", "mean")
    )
)

user_features = user_features.merge(user_time_to_product, on="user", how="left")

# Time to first add_to_cart por sessão -> média por usuário
session_first_add_to_cart = (
    df_user[df_user["is_add_to_cart"]]
    .groupby(["user", "session"], as_index=False)
    .agg(first_add_to_cart_time=("event_date", "min"))
)

session_time_to_add = session_duration.merge(
    session_first_add_to_cart,
    on=["user", "session"],
    how="left"
)

session_time_to_add["time_to_first_add_to_cart_sec"] = (
    session_time_to_add["first_add_to_cart_time"] -
    session_time_to_add["session_start"]
).dt.total_seconds()

user_time_to_add = (
    session_time_to_add
    .groupby("user", as_index=False)
    .agg(
        time_to_first_add_to_cart_sec=("time_to_first_add_to_cart_sec", "mean")
    )
)

user_features = user_features.merge(user_time_to_add, on="user", how="left")

# Product views before first add_to_cart por sessão -> soma por usuário
temp = df_user.merge(
    session_first_add_to_cart,
    on=["user", "session"],
    how="left"
)

product_before_add = temp[
    temp["is_product_page"] &
    (temp["event_date"] < temp["first_add_to_cart_time"])
]

product_views_before_add = (
    product_before_add
    .groupby("user")
    .size()
    .rename("product_views_before_add_to_cart")
    .reset_index()
)

user_features = user_features.merge(product_views_before_add, on="user", how="left")

user_features["product_views_before_add_to_cart"] = (
    user_features["product_views_before_add_to_cart"].fillna(0)
)

# Returned to search after first product na jornada do usuário
first_product_order = (
    df_user[df_user["is_product_page"]]
    .groupby("user")["user_event_order"]
    .min()
    .rename("first_product_order")
    .reset_index()
)

temp = df_user.merge(first_product_order, on="user", how="left")

returned_to_search = (
    temp[
        temp["is_search_page"] &
        (temp["user_event_order"] > temp["first_product_order"])
    ]
    .groupby("user")
    .size()
    .gt(0)
    .astype(int)
    .rename("returned_to_search")
    .reset_index()
)

user_features = user_features.merge(returned_to_search, on="user", how="left")
user_features["returned_to_search"] = user_features["returned_to_search"].fillna(0).astype(int)

# Status para Power BI
user_features["user_status"] = np.where(
    user_features["buyer"] == 1,
    "Buyer",
    "Non-Buyer"
)

# Converter boolean para 0/1
bool_cols = user_features.select_dtypes(include="bool").columns

user_features[bool_cols] = (
    user_features[bool_cols]
    .astype(int)
)

# Export
user_features.to_csv("user_features_powerbi.csv", index=False)

print("Arquivos gerados:")
print("- user_features_powerbi.csv")

user_features.head()

Arquivos gerados:
- user_features_powerbi.csv


,user,first_page,first_event_time,last_event_time,total_events,pageviews,unique_pages,page_changes,visited_search_listing_page,visited_listing_page,...,avg_session_duration_sec,avg_events_per_session,time_to_second_event_sec,second_event_same_session,avg_time_between_events,time_to_first_product_sec,time_to_first_add_to_cart_sec,product_views_before_add_to_cart,returned_to_search,user_status
0,10000015204044662882u,product_page,2022-10-02 22:33:00,2022-10-02 22:33:00,1,1,1,0,0,0,...,0.0,1.0,NaN,0,NaN,0.0,NaN,0.0,0,Non-Buyer
1,10000054772579221757u,product_page,2022-09-30 10:25:03,2022-09-30 10:25:03,1,1,1,0,0,0,...,0.0,1.0,NaN,0,NaN,0.0,NaN,0.0,0,Non-Buyer
2,10000059160930102536u,listing_page,2022-10-10 19:07:13,2022-10-10 19:09:21,3,3,3,2,1,1,...,128.0,3.0,30.0,1,64.0,128.0,NaN,0.0,0,Non-Buyer
3,10000070603737986791u,product_page,2022-10-01 20:09:10,2022-10-01 20:09:10,1,1,1,0,0,0,...,0.0,1.0,NaN,0,NaN,0.0,NaN,0.0,0,Non-Buyer
4,10000178367361043064u,product_page,2022-10-11 06:44:54,2022-10-11 06:44:54,1,1,1,0,0,0,...,0.0,1.0,NaN,0,NaN,0.0,NaN,0.0,0,Non-Buyer


In [11]:
session_features = pd.read_csv("user_features_powerbi.csv")

print(f"user_features_powerbi.csv: {len(session_features):,} linhas")



user_features_powerbi.csv: 288,088 linhas
